In [1]:
"""
ICA (FastICA) + GA Pipeline
Identical structure to SparsePCA pipeline:
  Section 1 : ICA alone  → SVM, RF, NB
  Section 2 : ICA + GA   → fitness matching classifier (SVM→SVM, RF→RF, NB→NB)

Table output format:
  Pipeline | Classifier | Precision | Recall | F1-Score | Accuracy | Runtime
"""

import time
import warnings
import numpy as np
import pandas as pd
import random

from sklearn.decomposition import FastICA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from deap import base, creator, tools, algorithms

warnings.filterwarnings("ignore")

# ── 1. Load data ──────────────────────────────────────────────────────────────
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── 2. ICA Configuration ──────────────────────────────────────────────────────
N_COMPONENTS = 12       # change to 53 or 12 to replicate other rows
MAX_ITER     = 1000     # ICA is iterative — needs enough iterations to converge
RANDOM_STATE = 42

print(f"N_COMPONENTS = {N_COMPONENTS}")
print(f"MAX_ITER     = {MAX_ITER}")
print()

# ── 3. Fit ICA ────────────────────────────────────────────────────────────────
# Important: fit ONLY on training data, then transform both
# ICA is not deterministic by nature but random_state pins the initialisation
print(f"Fitting FastICA (n_components={N_COMPONENTS}) ...")
t0 = time.time()

ica = FastICA(
    n_components=N_COMPONENTS,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    whiten='unit-variance'   # standard preprocessing step for ICA
)

X_train_np = X_train_raw.values if hasattr(X_train_raw, 'values') else X_train_raw
X_test_np  = X_test_raw.values  if hasattr(X_test_raw,  'values') else X_test_raw

train_ica = ica.fit_transform(X_train_np)
test_ica  = ica.transform(X_test_np)

ica_fit_time = time.time() - t0
print(f"ICA fit done in {ica_fit_time:.1f}s")
print(f"Output shape — Train: {train_ica.shape}, Test: {test_ica.shape}\n")

# ── 4. Helper: evaluate classifier, return result row ────────────────────────
def evaluate(pipeline_name, clf_name, clf, X_tr, X_te, y_tr, y_te, runtime_str):
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    return {
        "Pipeline"  : pipeline_name,
        "Classifier": clf_name,
        "Precision" : round(precision_score(y_te, preds, average='weighted'), 2),
        "Recall"    : round(recall_score   (y_te, preds, average='weighted'), 2),
        "F1-Score"  : round(f1_score       (y_te, preds, average='weighted'), 2),
        "Accuracy"  : f"{accuracy_score(y_te, preds)*100:.2f}%",
        "Runtime"   : runtime_str,
    }

# ── 5. SECTION 1 — ICA alone ──────────────────────────────────────────────────
print("=" * 60)
print("SECTION 1 : ICA alone")
print("=" * 60)

classifiers = {
    "SVM": SVC(random_state=42),
    "RF" : RandomForestClassifier(random_state=42),
    "NB" : GaussianNB(),
}

section1_rows = []
for clf_name, clf in classifiers.items():
    t0  = time.time()
    clf.fit(train_ica, y_train)
    preds = clf.predict(test_ica)
    elapsed = time.time() - t0
    rt_str = f"{elapsed:.1f} sec"

    row = {
        "Pipeline"  : "ICA",
        "Classifier": clf_name,
        "Precision" : round(precision_score(y_test, preds, average='weighted'), 2),
        "Recall"    : round(recall_score   (y_test, preds, average='weighted'), 2),
        "F1-Score"  : round(f1_score       (y_test, preds, average='weighted'), 2),
        "Accuracy"  : f"{accuracy_score(y_test, preds)*100:.2f}%",
        "Runtime"   : rt_str,
    }
    section1_rows.append(row)
    print(f"  {clf_name:4s} → Accuracy {row['Accuracy']}  ({rt_str})")

df1 = pd.DataFrame(section1_rows)[
    ["Pipeline","Classifier","Precision","Recall","F1-Score","Accuracy","Runtime"]
]
print()
print(df1.to_string(index=False))

# ── 6. GA helper ──────────────────────────────────────────────────────────────
def run_ga(X_tr, y_tr, fitness_clf, n_components):
    """
    Binary GA over ICA components.
    Each individual is a bit-vector of length n_components.
    1 = keep this ICA component, 0 = drop it.
    Fitness = mean StratifiedKFold CV accuracy of fitness_clf.
    """
    # Clean up DEAP registry to allow multiple calls
    for name in ("FitnessMax", "Individual"):
        if name in creator.__dict__:
            delattr(creator, name)

    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    tb = base.Toolbox()
    tb.register("attr_bool",  random.randint, 0, 1)
    tb.register("individual", tools.initRepeat, creator.Individual,
                tb.attr_bool, n=n_components)
    tb.register("population", tools.initRepeat, list, tb.individual)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def fitness_fn(individual):
        selected = [i for i, v in enumerate(individual) if v == 1]
        if len(selected) < 2:
            return (0.0,)
        scores = cross_val_score(fitness_clf, X_tr[:, selected], y_tr,
                                 cv=skf, scoring='accuracy')
        return (scores.mean(),)

    tb.register("evaluate", fitness_fn)
    tb.register("mate",     tools.cxTwoPoint)
    tb.register("mutate",   tools.mutFlipBit, indpb=0.05)
    tb.register("select",   tools.selTournament, tournsize=3)

    pop  = tb.population(n=100)
    hof  = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("max", np.max)

    algorithms.eaSimple(pop, tb,
                        cxpb=0.7, mutpb=0.2, ngen=50,
                        stats=stats, halloffame=hof, verbose=False)

    best     = hof[0]
    selected = [i for i, v in enumerate(best) if v == 1]
    return selected

# ── 7. SECTION 2 — ICA + GA ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 2 : ICA-GA  (fitness matching classifier)")
print("=" * 60)

fitness_classifiers = {
    "SVM": SVC(random_state=42),
    "RF" : RandomForestClassifier(random_state=42),
    "NB" : GaussianNB(),
}

section2_rows = []

for fit_name, fit_clf in fitness_classifiers.items():
    print(f"\n  → GA with {fit_name} fitness function ...")
    t0 = time.time()

    selected = run_ga(train_ica, y_train, fit_clf, N_COMPONENTS)
    ga_time  = time.time() - t0

    rt_str = f"{ga_time:.0f} sec" if ga_time < 60 else f"{ga_time/60:.0f} min"

    X_tr_sel = train_ica[:, selected]
    X_te_sel = test_ica [:, selected]

    # Fresh instance of same classifier type
    clf = fit_clf.__class__(**fit_clf.get_params())
    row = evaluate("ICA-GA", fit_name, clf,
                   X_tr_sel, X_te_sel, y_train, y_test, rt_str)

    section2_rows.append(row)
    print(f"     Selected {len(selected)}/{N_COMPONENTS} components | "
          f"Accuracy {row['Accuracy']} | Runtime {rt_str}")

df2 = pd.DataFrame(section2_rows)[
    ["Pipeline","Classifier","Precision","Recall","F1-Score","Accuracy","Runtime"]
]
print()
print(df2.to_string(index=False))

# ── 8. Full combined table ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"FULL TABLE  (N_COMPONENTS={N_COMPONENTS})")
print("=" * 60)
df_full = pd.concat([df1, df2], ignore_index=True)
print(df_full.to_string(index=False))

N_COMPONENTS = 12
MAX_ITER     = 1000

Fitting FastICA (n_components=12) ...
ICA fit done in 0.1s
Output shape — Train: (354, 12), Test: (89, 12)

SECTION 1 : ICA alone
  SVM  → Accuracy 88.76%  (0.0 sec)
  RF   → Accuracy 82.02%  (0.2 sec)
  NB   → Accuracy 82.02%  (0.0 sec)

Pipeline Classifier  Precision  Recall  F1-Score Accuracy Runtime
     ICA        SVM       0.90    0.89      0.89   88.76% 0.0 sec
     ICA         RF       0.86    0.82      0.83   82.02% 0.2 sec
     ICA         NB       0.84    0.82      0.82   82.02% 0.0 sec

SECTION 2 : ICA-GA  (fitness matching classifier)

  → GA with SVM fitness function ...
     Selected 11/12 components | Accuracy 91.01% | Runtime 3 min

  → GA with RF fitness function ...
     Selected 10/12 components | Accuracy 80.90% | Runtime 43 min

  → GA with NB fitness function ...
     Selected 9/12 components | Accuracy 83.15% | Runtime 46 sec

Pipeline Classifier  Precision  Recall  F1-Score Accuracy Runtime
  ICA-GA        SVM       0.92  